# 03 - Update training sets with predicted weak positives

This notebook updates the first-iteration training collections by adding weak-positive genes predicted in the first iteration.

Existing genes in each training set are preserved. New positive genes are appended only when they belong in that set, and the same number of new negative genes is appended to keep each training set balanced.


## 1. Imports and configuration

All paths are relative to the repository root.


In [ ]:
from copy import deepcopy
from pathlib import Path
import pickle
import random

import numpy as np
import pandas as pd


RANDOM_SEED = 123

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PREPARED_LISTS_DIR = PROJECT_ROOT / "results" / "prepared_lists"
FIRST_ITERATION_DIR = PROJECT_ROOT / "results" / "first_iteration"

EXPRESSION_FILE = DATA_DIR / "boeck_tmm.csv"
STRONG_POSITIVES_FIRST_ITERATION_FILE = DATA_DIR / "strong_positives_first_iteration.csv"
WEAK_POSITIVES_FILE = DATA_DIR / "weak_positives.csv"
PREDICTED_WEAK_POSITIVES_FILE = FIRST_ITERATION_DIR / "02.2_genes_pos_unanimity_with_proba.csv"
FIRST_ITERATION_TRAINING_SETS_FILE = PREPARED_LISTS_DIR / "01.2_random_training_sets.pkl"
SECOND_ITERATION_TRAINING_SETS_FILE = PREPARED_LISTS_DIR / "03.1_random_training_sets_second_iteration.pkl"

PREPARED_LISTS_DIR.mkdir(parents=True, exist_ok=True)


## 2. Helper functions


In [ ]:
def load_expression_matrix(path):
    """Load the expression matrix using WormBase gene IDs as row names."""
    expression = pd.read_csv(path, index_col=1)
    expression = expression.iloc[:, 1:]

    if expression.index.has_duplicates:
        duplicated = expression.index[expression.index.duplicated()].unique()
        raise ValueError(f"Expression matrix contains {len(duplicated)} duplicated gene IDs.")

    return expression


def load_pickle(path):
    with path.open("rb") as file:
        return pickle.load(file)


def save_pickle(obj, path):
    with path.open("wb") as file:
        pickle.dump(obj, file)


def ordered_unique(values):
    """Return values without duplicates while preserving their original order."""
    return list(dict.fromkeys(values))


def excluded_complex_from_training_key(training_key):
    """Return the complex excluded by scX keys, or None for all_negX keys."""
    complex_by_subset = {
        "sc1": "I",
        "sc2": "II",
        "sc3": "III",
        "sc4": "IV",
        "sc5": "V",
    }
    subset_name = training_key.split("_")[0]
    return complex_by_subset.get(subset_name)


In [ ]:
def get_new_positive_genes(predicted_weak_positives, weak_positive_annotations):
    """Return selected predicted weak-positive WormBase IDs with their annotation rows."""
    if "Selected_As_Weak_Positive" in predicted_weak_positives.columns:
        selected_mask = (
            predicted_weak_positives["Selected_As_Weak_Positive"]
            .astype(str)
            .str.lower()
            .isin(["true", "1", "yes"])
        )
        selected_predictions = predicted_weak_positives[selected_mask].copy()
    else:
        selected_predictions = predicted_weak_positives.copy()

    predicted_gene_ids = ordered_unique(selected_predictions["Gene"].astype(str))

    annotated = weak_positive_annotations[
        weak_positive_annotations["gene_ID_worm"].astype(str).isin(predicted_gene_ids)
    ].copy()
    annotated = annotated.drop_duplicates(subset="gene_ID_worm", keep="first")

    missing_annotations = sorted(set(predicted_gene_ids).difference(annotated["gene_ID_worm"].astype(str)))
    if missing_annotations:
        print(f"Selected weak positives without annotations: {len(missing_annotations)}")

    return annotated

def sample_new_negative_genes(
    expression_genes,
    genes_to_exclude,
    n_genes,
    rng,
):
    """Sample negative genes without using any existing or positive gene."""
    candidates = [
        gene for gene in expression_genes
        if gene not in genes_to_exclude
    ]

    if len(candidates) < n_genes:
        raise ValueError(
            f"Not enough candidate negative genes available: {len(candidates)} < {n_genes}."
        )

    return rng.sample(candidates, n_genes)


## 3. Load inputs


In [ ]:
for path in [
    EXPRESSION_FILE,
    STRONG_POSITIVES_FIRST_ITERATION_FILE,
    WEAK_POSITIVES_FILE,
    PREDICTED_WEAK_POSITIVES_FILE,
    FIRST_ITERATION_TRAINING_SETS_FILE,
]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

expression = load_expression_matrix(EXPRESSION_FILE)
strong_positives_first_iteration = pd.read_csv(STRONG_POSITIVES_FIRST_ITERATION_FILE)
weak_positives = pd.read_csv(WEAK_POSITIVES_FILE)
predicted_weak_positives = pd.read_csv(PREDICTED_WEAK_POSITIVES_FILE)
training_sets = load_pickle(FIRST_ITERATION_TRAINING_SETS_FILE)

predicted_weak_annotations = get_new_positive_genes(
    predicted_weak_positives,
    weak_positives,
)

print(f"Expression matrix: {expression.shape[0]:,} genes x {expression.shape[1]:,} features")
print(f"First-iteration training collections: {len(training_sets):,}")
print(f"Predicted weak positives available for update: {len(predicted_weak_annotations):,}")


## 4. Update training sets


In [ ]:
expression_genes = expression.index.astype(str).tolist()
expression_gene_set = set(expression_genes)
strong_gene_set = set(strong_positives_first_iteration["gene_ID_worm"].astype(str))
new_positive_complex = dict(
    zip(
        predicted_weak_annotations["gene_ID_worm"].astype(str),
        predicted_weak_annotations["Complex"].astype(str),
    )
)
new_positive_gene_ids = ordered_unique(predicted_weak_annotations["gene_ID_worm"].astype(str))

updated_training_sets = deepcopy(training_sets)
update_summary = []

for index, (training_key, subset) in enumerate(updated_training_sets.items(), start=1):
    excluded_complex = excluded_complex_from_training_key(training_key)
    train = subset["train"]

    old_genes = [str(gene) for gene in train["genes"]]
    old_classes = [int(label) for label in train["classes"]]
    old_gene_set = set(old_genes)
    old_positive_set = {gene for gene, label in zip(old_genes, old_classes) if label == 1}

    genes_to_add_as_positive = []
    for gene in new_positive_gene_ids:
        if gene in old_gene_set:
            continue
        if gene not in expression_gene_set:
            continue
        if excluded_complex is not None and new_positive_complex.get(gene) == excluded_complex:
            continue
        genes_to_add_as_positive.append(gene)

    known_positive_genes = strong_gene_set.union(new_positive_gene_ids)
    genes_to_exclude_from_negatives = set(old_genes).union(known_positive_genes)

    rng = random.Random(RANDOM_SEED + index)
    genes_to_add_as_negative = sample_new_negative_genes(
        expression_genes=expression_genes,
        genes_to_exclude=genes_to_exclude_from_negatives,
        n_genes=len(genes_to_add_as_positive),
        rng=rng,
    )

    updated_genes = old_genes + genes_to_add_as_positive + genes_to_add_as_negative
    updated_classes = old_classes + [1] * len(genes_to_add_as_positive) + [0] * len(genes_to_add_as_negative)

    train["genes"] = updated_genes
    train["classes"] = updated_classes
    train["data"] = expression.loc[updated_genes].to_numpy()

    update_summary.append({
        "training_set": training_key,
        "excluded_complex": excluded_complex if excluded_complex is not None else "None",
        "previous_positive_genes": len(old_positive_set),
        "added_positive_genes": len(genes_to_add_as_positive),
        "added_negative_genes": len(genes_to_add_as_negative),
        "final_positive_genes": sum(updated_classes),
        "final_negative_genes": len(updated_classes) - sum(updated_classes),
        "final_total_genes": len(updated_genes),
    })

update_summary = pd.DataFrame(update_summary)
update_summary.head()


## 5. Save updated training sets


In [ ]:
save_pickle(updated_training_sets, SECOND_ITERATION_TRAINING_SETS_FILE)
print(f"Saved updated training sets to: {SECOND_ITERATION_TRAINING_SETS_FILE}")
update_summary
